# LLaDA -> Text Projector -> Qwen Training (CPU-Optimized)

This notebook mirrors the projector-training flow but is tuned for CPU runtime.

CPU defaults:
1. Small sample budget
2. Short generation lengths
3. Float32 + CPU device
4. Heuristic draft plans by default (can enable model drafts)
5. Non-inplace projector replacement for stable autograd

In [ ]:
import subprocess
import sys

packages = [
    'transformers==4.49.0',
    'datasets',
    'accelerate',
    'sentencepiece',
    'protobuf',
    'huggingface_hub',
    'tqdm',
    'numpy'
]

cmd = [sys.executable, '-m', 'pip', 'install', '-q'] + packages
print('Installing dependencies...')
subprocess.run(cmd, check=True)
print('Dependency installation completed.')

In [ ]:
print('CPU-optimized mode enabled.')

In [ ]:
from pathlib import Path
import os
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Models
ANSWER_MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
LLADA_MODEL_ID = 'GSAI-ML/LLaDA-8B-Instruct'

# CPU mode
FORCE_CPU = True
USE_MODEL_DRAFTS = False  # Keep False on CPU for speed.

# Small-data settings
TOTAL_TRAIN_SAMPLES = 20
PER_DATASET_TRAIN_SAMPLES = 4
MAX_TEST_SAMPLES = 8
MAX_LENGTH = 256

# Generation lengths
DRAFT_PLAN_MAX_NEW_TOKENS = 32
ANSWER_MAX_NEW_TOKENS = 32

ENABLE_OOM_FALLBACK = True
LIGHTLY_TUNE_QWEN = False

# Projector tuning
PROJECTOR_BOTTLENECK_DIM = 128
PROJECTOR_DROPOUT = 0.10
PROJECTOR_LEARNING_RATE = 5e-5

# Optimization
NUM_TRAIN_EPOCHS = 1
GRADIENT_ACCUMULATION_STEPS = 1
WEIGHT_DECAY = 0.0
LOGGING_STEPS = 5
SAVE_STEPS = 50

OUTPUT_ROOT = Path('/kaggle/working/dart_text_projector_cpu')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FINAL_DIR = OUTPUT_ROOT / 'final_merged'
FINAL_DIR.mkdir(parents=True, exist_ok=True)

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

print(f'Output root: {OUTPUT_ROOT}')
print(f'CPU mode: {FORCE_CPU}')
print(f'Use model drafts: {USE_MODEL_DRAFTS}')
print(f'TOTAL_TRAIN_SAMPLES={TOTAL_TRAIN_SAMPLES}')
print(f'Projector: bottleneck={PROJECTOR_BOTTLENECK_DIM}, dropout={PROJECTOR_DROPOUT}, lr={PROJECTOR_LEARNING_RATE}')

In [ ]:
import gc
import re
from typing import Dict, List, Tuple

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, set_seed
from transformers.models.llama.modeling_llama import LlamaRMSNorm
from torch import nn

set_seed(SEED)

ARC_QUESTION_PROMPT_TEMPLATE = """Question: {question}\n{choices_text}"""
ARC_QUESTION_POSTFIX = (
    "\nAnswer with a single letter (A, B, C, or D) and no explanation. "
    "Your answer should start with \"Answer: \" and be followed by the letter "
    "of the answer you choose. Do not include any other text in your response."
)

def free_cuda_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def llada_plan_prompt(question: str) -> str:
    return (
        'You are a careful planning assistant.\n'
        'Given the math question, produce only a concise step-by-step PLAN.\n'
        'Do not output the final numeric answer.\n\n'
        f'Question: {question}\n\n'
        'Plan:'
    )

def heuristic_plan(question: str) -> str:
    return (
        '1) Identify known quantities and target.\n'
        '2) Write the equation or arithmetic steps.\n'
        '3) Compute carefully and keep a final numeric answer.'
    )

def build_qwen_prompt_parts(question: str) -> Tuple[str, str]:
    prefix = (
        'You are a math solver.\n'
        'Use the refined plan to solve the question.\n'
        'Return only the final numeric answer.\n\n'
        f'Question: {question}\n\n'
        'Refined plan:\n'
    )
    suffix = '\n\nAnswer:'
    return prefix, suffix

def extract_last_number(text: str) -> str:
    nums = re.findall(r'[-+]?\d+[\d,]*(?:\.\d+)?', text)
    return nums[-1].replace(',', '') if nums else ''

def normalize_answer_text(answer) -> str:
    if answer is None:
        return ''
    return str(answer).strip().replace(',', '')

def prepare_arc_sample(item: Dict) -> Dict[str, str]:
    question = item['question']
    choices = item['choices']
    choices_text = '\n'.join([f"{label}. {text}" for label, text in zip(choices['label'], choices['text'])])
    input_text = ARC_QUESTION_PROMPT_TEMPLATE.format(question=question, choices_text=choices_text) + ARC_QUESTION_POSTFIX
    answer_key = normalize_answer_text(item['answerKey'])
    return {'question': input_text, 'gold_final': answer_key}

def prepare_dart_sample(item: Dict) -> Dict[str, str]:
    question = item['query']
    answer_key = normalize_answer_text(item['gt_ans'])
    return {'question': f'Question: {question}', 'gold_final': answer_key}

def _uniform_sample_indices(total: int, sample_count: int, rng: random.Random) -> List[int]:
    if total <= 0:
        return []
    if total >= sample_count:
        return rng.sample(range(total), sample_count)
    return [rng.randrange(total) for _ in range(sample_count)]

def build_uniform_mixed_train_set(per_dataset_samples: int, seed: int) -> List[Dict[str, str]]:
    rng = random.Random(seed)
    mixed_records: List[Dict[str, str]] = []

    dataset_defs: List[Tuple[str, str, str, int]] = [
        ('arc_easy', 'allenai/ai2_arc', 'ARC-Easy', 0),
        ('arc_challenge', 'allenai/ai2_arc', 'ARC-Challenge', 0),
        ('dart_1', 'hkust-nlp/dart-math-pool-math', '', 1),
        ('dart_2', 'hkust-nlp/dart-math-pool-math', '', 2),
        ('dart_3', 'hkust-nlp/dart-math-pool-math', '', 3),
        ('dart_4', 'hkust-nlp/dart-math-pool-math', '', 4),
        ('dart_5', 'hkust-nlp/dart-math-pool-math', '', 5),
    ]

    for name, dataset_id, config_name, dart_level in dataset_defs:
        if name.startswith('arc'):
            ds = load_dataset(dataset_id, config_name, split='train')
            indices = _uniform_sample_indices(len(ds), per_dataset_samples, rng)
            for idx in indices:
                mixed_records.append(prepare_arc_sample(ds[int(idx)]))
        else:
            ds = load_dataset(dataset_id, split='train')
            ds = ds.filter(lambda x: x['query_metadata']['level'] == dart_level)
            indices = _uniform_sample_indices(len(ds), per_dataset_samples, rng)
            for idx in indices:
                mixed_records.append(prepare_dart_sample(ds[int(idx)]))

    rng.shuffle(mixed_records)
    return mixed_records

def build_draft_generator():
    if not USE_MODEL_DRAFTS:
        return None

    tokenizer = AutoTokenizer.from_pretrained(LLADA_MODEL_ID, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        LLADA_MODEL_ID,
        trust_remote_code=True,
        torch_dtype=torch.float32,
        device_map=None,
    )

    if hasattr(model, 'config') and hasattr(model.config, 'use_cache'):
        model.config.use_cache = False
    if hasattr(model, 'generation_config') and model.generation_config is not None:
        model.generation_config.use_cache = False

    return pipeline(
        'text-generation',
        model=model,
        tokenizer=tokenizer,
        do_sample=False,
        max_new_tokens=DRAFT_PLAN_MAX_NEW_TOKENS,
    )

def generate_draft_plan(question: str, draft_generator) -> str:
    if draft_generator is None:
        return heuristic_plan(question)

    prompt = llada_plan_prompt(question)
    try:
        out = draft_generator(prompt, return_full_text=False, use_cache=False)[0]['generated_text']
        out = out.strip()
        return out if out else heuristic_plan(question)
    except torch.cuda.OutOfMemoryError:
        if not ENABLE_OOM_FALLBACK:
            raise
        free_cuda_memory()
        return heuristic_plan(question)

class PlanProjectorTrainer(nn.Module):
    def __init__(self, lightly_tune_qwen: bool = False):
        super().__init__()
        self.model_dtype = torch.float32

        self.tokenizer = AutoTokenizer.from_pretrained(ANSWER_MODEL_ID, trust_remote_code=True)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        self.qwen = AutoModelForCausalLM.from_pretrained(
            ANSWER_MODEL_ID,
            trust_remote_code=True,
            torch_dtype=self.model_dtype,
        )

        self.device = torch.device('cpu') if FORCE_CPU else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.qwen = self.qwen.to(self.device)

        hidden_size = self.qwen.config.hidden_size
        self.qwen.text_projector = nn.Sequential(
            nn.Linear(hidden_size, PROJECTOR_BOTTLENECK_DIM),
            nn.GELU(approximate='tanh'),
            nn.Dropout(PROJECTOR_DROPOUT),
            nn.Linear(PROJECTOR_BOTTLENECK_DIM, PROJECTOR_BOTTLENECK_DIM),
            nn.GELU(approximate='tanh'),
            nn.Dropout(PROJECTOR_DROPOUT),
            nn.Linear(PROJECTOR_BOTTLENECK_DIM, hidden_size),
            LlamaRMSNorm(hidden_size, eps=self.qwen.config.rms_norm_eps),
        ).to(device=self.device, dtype=self.model_dtype)

        with torch.no_grad():
            nn.init.xavier_uniform_(self.qwen.text_projector[0].weight)
            nn.init.zeros_(self.qwen.text_projector[0].bias)
            nn.init.xavier_uniform_(self.qwen.text_projector[3].weight)
            nn.init.zeros_(self.qwen.text_projector[3].bias)
            nn.init.xavier_uniform_(self.qwen.text_projector[6].weight)
            nn.init.zeros_(self.qwen.text_projector[6].bias)

        self._configure_trainable_params(lightly_tune_qwen=lightly_tune_qwen)

    def _configure_trainable_params(self, lightly_tune_qwen: bool) -> None:
        for param in self.qwen.parameters():
            param.requires_grad = False

        for param in self.qwen.text_projector.parameters():
            param.requires_grad = True

        if lightly_tune_qwen:
            if hasattr(self.qwen, 'lm_head'):
                for param in self.qwen.lm_head.parameters():
                    param.requires_grad = True

    def trainable_parameters(self):
        return [p for p in self.qwen.parameters() if p.requires_grad]

    def _tokenize_no_special(self, text: str, max_len: int) -> torch.Tensor:
        ids = self.tokenizer(
            text,
            add_special_tokens=False,
            truncation=True,
            max_length=max_len,
            return_tensors='pt',
        )['input_ids']
        return ids.to(self.device)

    def compute_loss(self, question: str, draft_plan: str, gold_final: str, max_length: int) -> torch.Tensor:
        prefix, suffix = build_qwen_prompt_parts(question)
        target = gold_final.strip() + self.tokenizer.eos_token

        prefix_ids = self._tokenize_no_special(prefix, max_length)
        draft_ids = self._tokenize_no_special(draft_plan, max_length)
        suffix_ids = self._tokenize_no_special(suffix, max_length)
        target_ids = self._tokenize_no_special(target, max_length)

        max_total = max_length
        keep_target = min(target_ids.shape[1], max_total)
        target_ids = target_ids[:, -keep_target:]

        available_context = max_total - keep_target
        if available_context < 1:
            available_context = 1

        context_ids = torch.cat([prefix_ids, draft_ids, suffix_ids], dim=1)
        if context_ids.shape[1] > available_context:
            context_ids = context_ids[:, -available_context:]

        prefix_len = min(prefix_ids.shape[1], context_ids.shape[1])
        draft_len = min(draft_ids.shape[1], max(context_ids.shape[1] - prefix_len, 0))

        embeddings = self.qwen.get_input_embeddings()
        context_embeds = embeddings(context_ids)

        if draft_len > 0:
            draft_start = prefix_len
            draft_end = min(draft_start + draft_len, context_embeds.shape[1])
            draft_embeds = context_embeds[:, draft_start:draft_end, :]
            projected = self.qwen.text_projector(draft_embeds)
            before = context_embeds[:, :draft_start, :]
            after = context_embeds[:, draft_end:, :]
            context_embeds = torch.cat([before, projected, after], dim=1)

        target_embeds = embeddings(target_ids)
        inputs_embeds = torch.cat([context_embeds, target_embeds], dim=1)

        attention_mask = torch.ones(inputs_embeds.shape[:2], device=self.device, dtype=torch.long)
        labels = torch.full((1, inputs_embeds.shape[1]), -100, device=self.device, dtype=torch.long)
        labels[:, context_embeds.shape[1]:] = target_ids

        outputs = self.qwen(inputs_embeds=inputs_embeds, attention_mask=attention_mask, labels=labels)
        return outputs.loss

    @torch.no_grad()
    def generate_answer(self, question: str, draft_plan: str, max_length: int, max_new_tokens: int) -> str:
        self.qwen.eval()
        prefix, suffix = build_qwen_prompt_parts(question)

        prefix_ids = self._tokenize_no_special(prefix, max_length)
        draft_ids = self._tokenize_no_special(draft_plan, max_length)
        suffix_ids = self._tokenize_no_special(suffix, max_length)

        context_ids = torch.cat([prefix_ids, draft_ids, suffix_ids], dim=1)
        if context_ids.shape[1] > max_length:
            context_ids = context_ids[:, -max_length:]

        prefix_len = min(prefix_ids.shape[1], context_ids.shape[1])
        draft_len = min(draft_ids.shape[1], max(context_ids.shape[1] - prefix_len, 0))

        embeddings = self.qwen.get_input_embeddings()
        context_embeds = embeddings(context_ids)

        if draft_len > 0:
            draft_start = prefix_len
            draft_end = min(draft_start + draft_len, context_embeds.shape[1])
            draft_embeds = context_embeds[:, draft_start:draft_end, :]
            projected = self.qwen.text_projector(draft_embeds)
            before = context_embeds[:, :draft_start, :]
            after = context_embeds[:, draft_end:, :]
            context_embeds = torch.cat([before, projected, after], dim=1)

        output_ids = self.qwen.generate(
            inputs_embeds=context_embeds,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=self.tokenizer.pad_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
        )
        gen_only = output_ids[:, context_embeds.shape[1]:]
        return self.tokenizer.decode(gen_only[0], skip_special_tokens=True).strip()

print('Building 7-dataset uniform train set...')
train_seed_records = build_uniform_mixed_train_set(PER_DATASET_TRAIN_SAMPLES, SEED)
eval_seed_records = train_seed_records[: min(MAX_TEST_SAMPLES, len(train_seed_records))] if MAX_TEST_SAMPLES > 0 else []
print(f'Train seed samples: {len(train_seed_records)} | Eval seed samples: {len(eval_seed_records)}')

print('Loading draft generator...')
draft_generator = build_draft_generator()

train_records: List[Dict[str, str]] = []
eval_records: List[Dict[str, str]] = []

print('Generating draft plans...')
for ex in train_seed_records:
    draft_plan = generate_draft_plan(ex['question'], draft_generator)
    train_records.append({'question': ex['question'], 'draft_plan': draft_plan, 'gold_final': ex['gold_final']})

for ex in eval_seed_records:
    draft_plan = generate_draft_plan(ex['question'], draft_generator)
    eval_records.append({'question': ex['question'], 'draft_plan': draft_plan, 'gold_final': ex['gold_final']})

if draft_generator is not None:
    del draft_generator
free_cuda_memory()

# Exact small-data cap
if len(train_records) > TOTAL_TRAIN_SAMPLES:
    random.shuffle(train_records)
    train_records = train_records[:TOTAL_TRAIN_SAMPLES]
if len(eval_records) > MAX_TEST_SAMPLES:
    eval_records = eval_records[:MAX_TEST_SAMPLES]

print(f'After cap -> train: {len(train_records)} | eval: {len(eval_records)}')

print('Loading Qwen + text_projector...')
model_wrapper = PlanProjectorTrainer(lightly_tune_qwen=LIGHTLY_TUNE_QWEN)
trainable_count = sum(p.numel() for p in model_wrapper.trainable_parameters())
print(f'Trainable parameters: {trainable_count}')

In [ ]:
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Running device: {model_wrapper.device}')

In [ ]:
def train_projector(train_records):
    model_wrapper.qwen.train()
    optimizer = torch.optim.AdamW(
        model_wrapper.trainable_parameters(),
        lr=PROJECTOR_LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    global_step = 0
    losses = []
    running_loss = 0.0
    no_grad_fallback_count = 0

    for epoch in range(NUM_TRAIN_EPOCHS):
        random.shuffle(train_records)
        optimizer.zero_grad(set_to_none=True)

        for idx, rec in enumerate(train_records, start=1):
            with torch.enable_grad():
                loss = model_wrapper.compute_loss(
                    question=rec['question'],
                    draft_plan=rec['draft_plan'],
                    gold_final=rec['gold_final'],
                    max_length=MAX_LENGTH,
                )

            if not loss.requires_grad:
                reg = None
                for p in model_wrapper.qwen.text_projector.parameters():
                    term = (p.float() ** 2).mean()
                    reg = term if reg is None else (reg + term)
                loss = 1e-6 * reg
                no_grad_fallback_count += 1

            (loss / GRADIENT_ACCUMULATION_STEPS).backward()
            running_loss += loss.item()

            if idx % GRADIENT_ACCUMULATION_STEPS == 0 or idx == len(train_records):
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1

                step_loss = running_loss / GRADIENT_ACCUMULATION_STEPS
                losses.append(step_loss)
                running_loss = 0.0

                if global_step % LOGGING_STEPS == 0:
                    print(f'epoch={epoch + 1} step={global_step} loss={step_loss:.6f}')

                if global_step % SAVE_STEPS == 0:
                    ckpt_dir = OUTPUT_ROOT / f'checkpoint-{global_step}'
                    ckpt_dir.mkdir(parents=True, exist_ok=True)
                    torch.save(model_wrapper.qwen.text_projector.state_dict(), ckpt_dir / 'text_projector.pth')
                    print(f'Saved projector checkpoint to: {ckpt_dir / "text_projector.pth"}')

    mean_loss = float(np.mean(losses)) if losses else 0.0
    return {
        'global_steps': global_step,
        'mean_loss': mean_loss,
        'no_grad_fallback_count': no_grad_fallback_count,
    }

print('CPU projector trainer ready.')

In [ ]:
print('Starting projector training...')
train_stats = train_projector(train_records)

projector_path = FINAL_DIR / 'text_projector.pth'
torch.save(model_wrapper.qwen.text_projector.state_dict(), projector_path)
model_wrapper.tokenizer.save_pretrained(str(FINAL_DIR))

print('Training finished.')
print(f'Saved text projector to: {projector_path}')
print(train_stats)

In [ ]:
@torch.no_grad()
def quick_eval(records):
    correct = 0
    total = len(records)
    for ex in records:
        pred_text = model_wrapper.generate_answer(
            question=ex['question'],
            draft_plan=ex['draft_plan'],
            max_length=MAX_LENGTH,
            max_new_tokens=ANSWER_MAX_NEW_TOKENS,
        )
        pred_num = extract_last_number(pred_text)
        correct += int(pred_num == ex['gold_final'])
    return correct / total if total else 0.0

acc = quick_eval(eval_records)
print(f'Eval samples: {len(eval_records)}')
print(f'Exact-number accuracy: {acc:.3f}')
print(f'No-grad fallback count: {train_stats.get("no_grad_fallback_count", 0)}')

meta = {
    'force_cpu': FORCE_CPU,
    'use_model_drafts': USE_MODEL_DRAFTS,
    'total_train_samples': TOTAL_TRAIN_SAMPLES,
    'per_dataset_train_samples_before_cap': PER_DATASET_TRAIN_SAMPLES,
    'projector_bottleneck_dim': PROJECTOR_BOTTLENECK_DIM,
    'projector_dropout': PROJECTOR_DROPOUT,
    'projector_learning_rate': PROJECTOR_LEARNING_RATE,
    'num_train_records': len(train_records),
    'num_eval_records': len(eval_records),
    'lightly_tune_qwen': LIGHTLY_TUNE_QWEN,
    'train_global_steps': train_stats['global_steps'],
    'train_mean_loss': train_stats['mean_loss'],
    'no_grad_fallback_count': train_stats.get('no_grad_fallback_count', 0),
    'quick_eval_exact_number_accuracy': acc,
}
with open(OUTPUT_ROOT / 'training_meta.json', 'w', encoding='utf-8') as f:
    import json
    json.dump(meta, f, indent=2)

print(f'Meta saved to: {OUTPUT_ROOT / "training_meta.json"}')